# Driver Warning System - Model Training & Export

This notebook demonstrates the complete pipeline:
1. Generate synthetic driving traces
2. Extract features from sliding windows
3. Auto-label based on curvature
4. Train baseline RandomForest and Keras MLP
5. Evaluate models
6. Export to TensorFlow Lite
7. Test TFLite inference

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import our modules
sys.path.insert(0, '..')
from python.simulate import generate_trip
from python.feature_extractor import (
    sliding_windows,
    extract_features_from_windows,
    fit_scaler,
    apply_scaler,
    save_scaler,
    FEATURE_NAMES
)
from python.train import auto_label_windows, build_keras_model, convert_to_tflite, save_feature_schema

import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Imports complete!")

## 1. Generate Synthetic Driving Data

In [ ]:
# Generate synthetic trip
print("Generating synthetic driving traces...")
trip_df = generate_trip(
    duration_sec=600,  # 10 minutes
    num_curves=8,      # Mix of curves
    sample_rate_hz=10.0
)

print(f"Generated {len(trip_df)} samples")
print(f"Duration: {len(trip_df)/10:.1f} seconds")
print(f"\nFirst few rows:")
trip_df.head()

In [ ]:
# Visualize the trip
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# GPS trajectory
axes[0, 0].plot(trip_df['lon'], trip_df['lat'], 'b-', alpha=0.6, linewidth=2)
axes[0, 0].scatter(trip_df['lon'].iloc[0], trip_df['lat'].iloc[0], c='green', s=100, label='Start', zorder=5)
axes[0, 0].scatter(trip_df['lon'].iloc[-1], trip_df['lat'].iloc[-1], c='red', s=100, label='End', zorder=5)
axes[0, 0].set_xlabel('Longitude')
axes[0, 0].set_ylabel('Latitude')
axes[0, 0].set_title('GPS Trajectory')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Speed over time
axes[0, 1].plot(trip_df.index / 10, trip_df['speed'], 'g-', alpha=0.7)
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Speed (m/s)')
axes[0, 1].set_title('Speed Profile')
axes[0, 1].grid(True, alpha=0.3)

# Lateral acceleration
axes[1, 0].plot(trip_df.index / 10, trip_df['acc_y'], 'r-', alpha=0.7)
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Lateral Acceleration (m/s²)')
axes[1, 0].set_title('Lateral Acceleration (acc_y)')
axes[1, 0].grid(True, alpha=0.3)

# Yaw rate
axes[1, 1].plot(trip_df.index / 10, trip_df['gyro_z'], 'm-', alpha=0.7)
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Yaw Rate (deg/s)')
axes[1, 1].set_title('Yaw Rate (gyro_z)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Speed range: {trip_df['speed'].min():.1f} - {trip_df['speed'].max():.1f} m/s")
print(f"Gyro_z range: {trip_df['gyro_z'].min():.1f} - {trip_df['gyro_z'].max():.1f} deg/s")
print(f"Acc_y range: {trip_df['acc_y'].min():.1f} - {trip_df['acc_y'].max():.1f} m/s²")

## 2. Extract Features from Sliding Windows

In [ ]:
# Generate sliding windows
print("Generating sliding windows (3s window, 1s step)...")
windows = sliding_windows(trip_df, window_sec=3.0, step_sec=1.0, sample_rate_hz=10.0)
print(f"Generated {len(windows)} windows")
print(f"Samples per window: {len(windows[0])}")

In [ ]:
# Extract features
print("Extracting features from windows...")
features_df = extract_features_from_windows(windows, points_ahead=5)
print(f"Feature matrix shape: {features_df.shape}")
print(f"\nFeature names (in order):")
for i, name in enumerate(features_df.columns, 1):
    print(f"  {i}. {name}")

print(f"\nFeature statistics:")
features_df.describe()

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feature in enumerate(FEATURE_NAMES):
    axes[i].hist(features_df[feature], bins=30, alpha=0.7, edgecolor='black')
    axes[i].set_title(feature)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Auto-Label Windows Based on Curvature

In [ ]:
# Auto-label based on curve radius
labels = auto_label_windows(features_df)

label_map = {0: "safe", 1: "mild", 2: "urgent", 3: "hectic"}
label_counts = pd.Series(labels).value_counts().sort_index()

print("Label distribution:")
for label, count in label_counts.items():
    print(f"  {label} ({label_map[label]:7s}): {count:4d} ({count/len(labels)*100:5.1f}%)")

# Visualize label distribution
plt.figure(figsize=(10, 6))
plt.bar([label_map[i] for i in label_counts.index], label_counts.values, 
        color=['green', 'yellow', 'orange', 'red'], alpha=0.7, edgecolor='black')
plt.xlabel('Severity Class')
plt.ylabel('Count')
plt.title('Distribution of Auto-Labeled Windows')
plt.grid(True, alpha=0.3, axis='y')
plt.show()

## 4. Train/Val/Test Split & Scaling

In [ ]:
# Split data
X = features_df.values
y = labels

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train set: {len(X_train)} samples")
print(f"Val set:   {len(X_val)} samples")
print(f"Test set:  {len(X_test)} samples")

In [ ]:
# Fit scaler on training data
scaler = fit_scaler(pd.DataFrame(X_train, columns=FEATURE_NAMES))

X_train_scaled = apply_scaler(X_train, scaler)
X_val_scaled = apply_scaler(X_val, scaler)
X_test_scaled = apply_scaler(X_test, scaler)

print("Scaler fitted and applied")
print(f"Scaled train mean: {X_train_scaled.mean(axis=0).round(3)}")
print(f"Scaled train std:  {X_train_scaled.std(axis=0).round(3)}")

## 5. Train Baseline RandomForest Model

In [ ]:
# Train RandomForest baseline
print("Training RandomForest baseline...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# Evaluate
rf_pred = rf_model.predict(X_test_scaled)
print("\nRandomForest Classification Report:")
print(classification_report(y_test, rf_pred, target_names=["safe", "mild", "urgent", "hectic"]))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': FEATURE_NAMES,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], alpha=0.7, edgecolor='black')
plt.xlabel('Importance')
plt.title('RandomForest Feature Importance')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 6. Train Keras MLP Model

In [ ]:
# Build Keras model
print("Building Keras MLP...")
keras_model = build_keras_model(input_dim=len(FEATURE_NAMES), num_classes=4)
keras_model.summary()

In [ ]:
# Train with early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print("Training Keras model...")
history = keras_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluate Keras Model

In [ ]:
# Predictions
y_pred_probs = keras_model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Keras MLP Classification Report:")
print(classification_report(y_test, y_pred, target_names=["safe", "mild", "urgent", "hectic"]))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["safe", "mild", "urgent", "hectic"])
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Keras MLP', fontsize=14)
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, label_name in enumerate(["safe", "mild", "urgent", "hectic"]):
    if cm[i].sum() > 0:
        acc = cm[i, i] / cm[i].sum()
        print(f"  {label_name:7s}: {acc*100:.1f}%")

## 8. Export to TensorFlow Lite

In [ ]:
# Save scaler
save_scaler(scaler, "../models/scaler.pkl")

# Save feature schema
save_feature_schema("../models/feature_schema.json")

# Convert to TFLite
convert_to_tflite(keras_model, "../models/curve_detector.tflite")

## 9. Test TFLite Inference

In [ ]:
# Load TFLite model
interpreter = tf.lite.Interpreter(model_path="../models/curve_detector.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Model Details:")
print(f"  Input shape:  {input_details[0]['shape']}")
print(f"  Input dtype:  {input_details[0]['dtype']}")
print(f"  Output shape: {output_details[0]['shape']}")
print(f"  Output dtype: {output_details[0]['dtype']}")

In [ ]:
# Test inference on sample data
print("\nTesting TFLite inference on 10 samples:\n")

for i in range(10):
    # Prepare input
    input_data = X_test_scaled[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    
    # Run inference
    interpreter.invoke()
    
    # Get output
    output_data = interpreter.get_tensor(output_details[0]['index'])
    predicted_class = np.argmax(output_data[0])
    
    print(f"Sample {i+1}: True={label_map[y_test[i]]}, Pred={label_map[predicted_class]}, Probs={output_data[0].round(3)}")

## 10. Generate Sample Predictions CSV

In [ ]:
# Create sample predictions DataFrame
n_samples = 20
sample_df = pd.DataFrame(X_test[:n_samples], columns=FEATURE_NAMES)
sample_df['true_label'] = [label_map[y] for y in y_test[:n_samples]]

# Get TFLite predictions
predictions = []
for i in range(n_samples):
    input_data = X_test_scaled[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    predictions.append(output_data[0])

pred_df = pd.DataFrame(predictions, columns=['prob_safe', 'prob_mild', 'prob_urgent', 'prob_hectic'])
sample_df = pd.concat([sample_df, pred_df], axis=1)

# Save to CSV
sample_df.to_csv("../models/predictions_sample.csv", index=False)
print("Sample predictions saved to: ../models/predictions_sample.csv")
print("\nFirst few rows:")
sample_df.head(10)

## Summary

✅ Generated synthetic driving traces with mixed curve severities  
✅ Extracted 9 features from 3-second sliding windows  
✅ Auto-labeled windows based on curvature radius  
✅ Trained RandomForest baseline model  
✅ Trained Keras MLP model  
✅ Evaluated models with classification reports and confusion matrices  
✅ Exported Keras model to TensorFlow Lite format  
✅ Tested TFLite inference  
✅ Saved model artifacts (TFLite, scaler, feature schema, sample predictions)  

**Next steps:**
- Integrate TFLite model into Android app
- Test on real device with sensor data
- Tune alert thresholds and haptic patterns